# Elastic Net (EN) – L1 + L2 Regularisierung
## Wind Power Forecasting | Seminar WIBA

Dieses Notebook implementiert Elastic Net Regression als Forecast-Modell für Windproduktion.

Elastic Net kombiniert:
- **LASSO (L1)**: erzeugt Sparsität – setzt unwichtige Koeffizienten auf 0
- **Ridge (L2)**: stabilisiert korrelierte Prädiktoren – verhindert instabile Koeffizienten

Besonders relevant wenn nach Signaldekomposition (z.B. EMD/VMD) viele korrelierte IMFs als Features vorliegen.

**Referenz:** Yang u. a. (2024) – optimaler Regularisierungspfad via LARS-EN mit AIC-Kriterium

---


## 1. Imports und Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import ElasticNet, ElasticNetCV, enet_path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline

# Reproduzierbarkeit
np.random.seed(42)

# Plot-Stil
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size']   = 11
sns.set_style("whitegrid")

print("Alle Imports erfolgreich.")


## 2. Theoretischer Hintergrund

### 2.1 Die Verlustfunktion

Elastic Net löst folgendes Minimierungsproblem:

$$\hat{\beta}^{EN} = \arg\min_{\beta} \left\{ \|y - U\beta\|^2 + \lambda_1 \sum_j |\beta_j| + \lambda_2 \sum_j \beta_j^2 \right\}$$

**Parameter:**
- $\lambda_1$: steuert L1-Strafe → Sparsität (feature selection)
- $\lambda_2$: steuert L2-Strafe → Stabilität bei korrelierten Features
- $l_1\_ratio = \frac{\lambda_1}{\lambda_1 + \lambda_2}$: Mischungsparameter

**Grenzfälle:**
- $l_1\_ratio = 1$ → reines LASSO
- $l_1\_ratio = 0$ → reines Ridge

### 2.2 Warum Elastic Net für Wind-IMFs?

Nach Empirical Mode Decomposition (EMD) entstehen *Intrinsic Mode Functions* (IMFs),
die als Prädiktoren hochgradig korreliert sein können. In diesem Fall:
- LASSO wählt willkürlich eine IMF aus einer korrelierten Gruppe
- Ridge behält alle, kann aber nicht selektieren
- **Elastic Net selektiert Gruppen korrelierter IMFs gemeinsam**

### 2.3 LARS-EN mit AIC-Kriterium (nach Yang u. a., 2024)

Der optimale Regularisierungspfad wird via *Least Angle Regression* für Elastic Net (LARS-EN) bestimmt.
Das **AIC-Kriterium** (Akaike Information Criterion) wählt das Modell mit bestem Bias-Varianz-Trade-off:

$$AIC = n \cdot \ln(RSS/n) + 2k$$

wobei $k$ die Anzahl nicht-null Koeffizienten ist.


## 3. Datenvorbereitung

### 3.1 Datensatz laden
Echte Winddaten aus `final_dataset.csv` (Schonungen 2016, stündlich aggregiert).


In [ ]:
# ── Echte Winddaten laden ─────────────────────────────────────────────────
DATA_PATH = '/Users/noahweis/wind_bidding_project/data/processed/final_dataset.csv'

df = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

print(f'Datensatz geladen: {df.shape[0]} Zeilen, {df.shape[1]} Spalten')
print(f'Zeitraum: {df.timestamp.min()} bis {df.timestamp.max()}')
print(f'\nSpalten:')
print(df.dtypes)
print(f'\nFehlende Werte:')
print(df.isnull().sum())
df.head()


### 3.2 Korrelationsstruktur der Features

Elastic Net ist besonders dort stark, wo Features untereinander korreliert sind –
z.B. benachbarte IMFs nach EMD-Zerlegung.


In [ ]:
# ── Features und Target definieren ──────────────────────────────────────────
# Automatisch alle numerischen Spalten außer timestamp und power als Features
TARGET = 'power'

# Alle verfügbaren numerischen Spalten anzeigen
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print('Verfügbare numerische Spalten:')
print(numeric_cols)

# Features: alle außer Target
# Passe diese Liste an eure tatsächlichen Spalten an!
FEATURES = [c for c in numeric_cols if c != TARGET]
print(f'\nFeatures ({len(FEATURES)}): {FEATURES}')
print(f'Target: {TARGET}')

# Deskriptive Statistik
df[FEATURES + [TARGET]].describe().round(3)


### 3.3 Train/Test Split

**Wichtig:** `shuffle=False` – bei Zeitreihen keine zufällige Aufteilung!


In [ ]:
X = df[FEATURES].values
y = df[TARGET].values

# Zeitreihen-konformer Split (keine zufällige Mischung!)
TEST_SIZE = 0.2
split_idx = int(len(X) * (1 - TEST_SIZE))

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Skalierung: PFLICHT für regularisierte Modelle
# EN bestraft alle Beta-Koeffizienten gleich → Features müssen gleiche Skala haben
scaler   = StandardScaler()
X_tr_sc  = scaler.fit_transform(X_train)
X_te_sc  = scaler.transform(X_test)

print(f"Train: {len(X_train)} Samples ({100*(1-TEST_SIZE):.0f}%)")
print(f"Test:  {len(X_test)} Samples ({100*TEST_SIZE:.0f}%)")
print(f"Features: {len(FEATURES)}")
print(f"\nSkalierung Check – Train mean ≈ 0: {X_tr_sc.mean(axis=0).round(3)}")


## 4. Elastic Net – Modell und Regularisierungspfad

### 4.1 Einfluss des l1_ratio (L1/L2-Mischung)

`l1_ratio = 1` → LASSO (maximale Sparsität)  
`l1_ratio = 0.5` → Elastic Net (Balance)  
`l1_ratio = 0` → Ridge (keine Selektion)


In [ ]:
# Regularisierungspfad für verschiedene l1_ratio visualisieren
alphas = np.logspace(-4, 2, 200)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
l1_ratios = [1.0, 0.5, 0.1]
titles    = ['LASSO (l1_ratio=1.0)', 'Elastic Net (l1_ratio=0.5)', 'Ridge-like (l1_ratio=0.1)']

for ax, l1r, title in zip(axes, l1_ratios, titles):
    alphas_path, coefs, _ = enet_path(X_tr_sc, y_train, l1_ratio=l1r, alphas=alphas)
    for i, feat in enumerate(FEATURES):
        ax.plot(np.log10(alphas_path), coefs[i], linewidth=1.5, label=feat)
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax.set_xlabel('log10(alpha)')
    ax.set_title(title, fontsize=10)
    ax.set_xlim([-4, 2])

axes[0].set_ylabel('Koeffizient β')
axes[0].legend(fontsize=7, loc='upper left')
fig.suptitle('Regularisierungspfade – LASSO vs. Elastic Net vs. Ridge-like', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('../results/figures/en_regularization_paths.png', dpi=150, bbox_inches='tight')
plt.show()
print("Beobachtung: LASSO setzt Koeffizienten abrupt auf 0, EN setzt korrelierte Gruppen gemeinsam.")


### 4.2 Modellselektion via AIC (nach Yang u. a., 2024)

AIC = n · ln(RSS/n) + 2k

- RSS: Residual Sum of Squares
- k: Anzahl nicht-null Koeffizienten (Modellkomplexität)
- Wählt automatisch das sparseste Modell mit gutem Fit


In [ ]:
def compute_aic(y_true, y_pred, n_nonzero):
    """
    AIC für lineares Modell.
    n_nonzero: Anzahl nicht-null Koeffizienten (= Modellkomplexität k)
    """
    n   = len(y_true)
    rss = np.sum((y_true - y_pred) ** 2)
    rss = max(rss, 1e-10)  # numerische Stabilität
    aic = n * np.log(rss / n) + 2 * n_nonzero
    return aic


# AIC-Pfad berechnen für l1_ratio = 0.5 (typischer EN-Wert)
L1_RATIO = 0.5
alphas_path, coefs_path, _ = enet_path(X_tr_sc, y_train, l1_ratio=L1_RATIO, alphas=alphas)

aic_values  = []
nnz_values  = []

for i, alpha in enumerate(alphas_path):
    coef_i  = coefs_path[:, i]
    y_pred_i = X_tr_sc @ coef_i
    nnz      = np.sum(np.abs(coef_i) > 1e-6)
    aic_i    = compute_aic(y_train, y_pred_i, nnz)
    aic_values.append(aic_i)
    nnz_values.append(nnz)

aic_values = np.array(aic_values)
nnz_values = np.array(nnz_values)

# Optimales Alpha nach AIC
best_idx   = np.argmin(aic_values)
best_alpha = alphas_path[best_idx]
best_nnz   = nnz_values[best_idx]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

ax1.plot(np.log10(alphas_path), aic_values, color='#C44E52', linewidth=2)
ax1.axvline(np.log10(best_alpha), color='navy', linestyle='--', label=f'Opt. alpha={best_alpha:.4f}')
ax1.set_ylabel('AIC')
ax1.set_title(f'AIC-Pfad (l1_ratio={L1_RATIO})')
ax1.legend()

ax2.step(np.log10(alphas_path), nnz_values, color='#55A868', linewidth=2)
ax2.axvline(np.log10(best_alpha), color='navy', linestyle='--')
ax2.set_ylabel('Nicht-null Koeffizienten')
ax2.set_xlabel('log10(alpha)')
ax2.set_title('Modellsparsität entlang des Pfades')

plt.tight_layout()
plt.savefig('../results/figures/en_aic_path.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Optimales alpha (AIC): {best_alpha:.6f}")
print(f"Nicht-null Koeffizienten: {best_nnz} von {len(FEATURES)}")


### 4.3 Finales Modell mit AIC-Regularisierung trainieren


In [ ]:
# Modell mit AIC-optimalem Alpha trainieren
en_aic = ElasticNet(alpha=best_alpha, l1_ratio=L1_RATIO, max_iter=10000)
en_aic.fit(X_tr_sc, y_train)

y_pred_aic = en_aic.predict(X_te_sc)

# Koeffizienten
coef_series = pd.Series(en_aic.coef_, index=FEATURES).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#C44E52' if c > 0 else '#4C72B0' for c in coef_series]
coef_series.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(f'Elastic Net Koeffizienten (AIC-optimal, alpha={best_alpha:.4f})')
ax.set_xlabel('Koeffizient β')
# Null-Koeffizienten markieren
for i, (feat, val) in enumerate(coef_series.items()):
    if abs(val) < 1e-6:
        ax.get_yticklabels()[i].set_color('gray')
        ax.get_yticklabels()[i].set_fontstyle('italic')
plt.tight_layout()
plt.savefig('../results/figures/en_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

nonzero = (np.abs(en_aic.coef_) > 1e-6).sum()
print(f"Aktive Features (β ≠ 0): {nonzero}/{len(FEATURES)}")
print("\nKoeffizienten:")
print(coef_series.round(4).to_string())


## 5. Kreuzvalidierung (ElasticNetCV)

Neben AIC ist Kreuzvalidierung eine Alternative zur Hyperparameter-Selektion.
Für Zeitreihen: **kein shuffle** in KFold!


In [ ]:
# ElasticNetCV mit zeitreihentauglicher KFold (kein Shuffle!)
kf = KFold(n_splits=5, shuffle=False)

en_cv = ElasticNetCV(
    l1_ratio  = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0],  # alle l1_ratio-Kandidaten
    alphas    = np.logspace(-4, 1, 50),
    cv        = kf,
    max_iter  = 10000,
    n_jobs    = -1,
)

en_cv.fit(X_tr_sc, y_train)
y_pred_cv = en_cv.predict(X_te_sc)

print(f"Optimales alpha (CV):    {en_cv.alpha_:.6f}")
print(f"Optimales l1_ratio (CV): {en_cv.l1_ratio_:.2f}")
print(f"Aktive Features:         {(np.abs(en_cv.coef_) > 1e-6).sum()}/{len(FEATURES)}")


## 6. Evaluation

### 6.1 Forecast-Metriken


In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mae_metric(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

def mape(y_true, y_pred, eps=1e-6):
    mask = np.abs(y_true) > eps
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

# Persistence Baseline
y_persistence = np.roll(y_test, 24)
y_persistence[:24] = y_train[-24:]

# Mean Baseline
y_mean = np.full_like(y_test, y_train.mean())

results = {
    'Persistence':     {'rmse': rmse(y_test, y_persistence), 'mae': mae_metric(y_test, y_persistence), 'mape': mape(y_test, y_persistence)},
    'Mean Baseline':   {'rmse': rmse(y_test, y_mean),        'mae': mae_metric(y_test, y_mean),        'mape': mape(y_test, y_mean)},
    'EN (AIC)':        {'rmse': rmse(y_test, y_pred_aic),    'mae': mae_metric(y_test, y_pred_aic),    'mape': mape(y_test, y_pred_aic)},
    'EN (CV)':         {'rmse': rmse(y_test, y_pred_cv),     'mae': mae_metric(y_test, y_pred_cv),     'mape': mape(y_test, y_pred_cv)},
}

df_results = pd.DataFrame(results).T.round(3)
print("── Forecast-Metriken ────────────────────────────────")
print(df_results.to_string())

# Tabelle speichern
df_results.to_csv('../results/tables/en_forecast_metrics.csv')


### 6.2 Forecast vs. Actual


In [ ]:
N_PLOT = 336  # 2 Wochen

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Zeitreihe
ax = axes[0]
ax.plot(y_test[:N_PLOT],        label='Actual',           alpha=0.9, linewidth=1.5)
ax.plot(y_pred_aic[:N_PLOT],    label='EN (AIC)',          alpha=0.8, linestyle='--', linewidth=1.5)
ax.plot(y_pred_cv[:N_PLOT],     label='EN (CV)',           alpha=0.8, linestyle=':',  linewidth=1.5)
ax.plot(y_persistence[:N_PLOT], label='Persistence',       alpha=0.5, linestyle='-.',  linewidth=1.0)
ax.set_title('Elastic Net Forecast – erste 2 Wochen Testperiode')
ax.set_xlabel('Stunde')
ax.set_ylabel('Leistung [kW]')
ax.legend()

# Residuenplot
ax2 = axes[1]
residuals_aic = y_test[:N_PLOT] - y_pred_aic[:N_PLOT]
residuals_cv  = y_test[:N_PLOT] - y_pred_cv[:N_PLOT]
ax2.plot(residuals_aic, label='Residuen EN (AIC)', alpha=0.7, linewidth=1.0)
ax2.plot(residuals_cv,  label='Residuen EN (CV)',  alpha=0.7, linewidth=1.0, linestyle='--')
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_title('Residuen')
ax2.set_xlabel('Stunde')
ax2.set_ylabel('Fehler [kW]')
ax2.legend()

plt.tight_layout()
plt.savefig('../results/figures/en_forecast_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_pred, title in zip(axes, [y_pred_aic, y_pred_cv], ['EN (AIC)', 'EN (CV)']):
    ax.scatter(y_test, y_pred, alpha=0.2, s=5, color='#4C72B0')
    lim = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lim, lim, 'r--', linewidth=1.5, label='Perfekte Prognose')
    ax.set_xlabel('Actual [kW]')
    ax.set_ylabel('Predicted [kW]')
    ax.set_title(f'{title} – Actual vs. Predicted\nRMSE={rmse(y_test,y_pred):.1f}  MAE={mae_metric(y_test,y_pred):.1f}')
    ax.legend()

plt.tight_layout()
plt.savefig('../results/figures/en_scatter.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Verbindung zum Bidding-Modell (Newsvendor)

Elastic Net liefert einen **Punkt-Forecast** → muss in Gebot überführt werden.

### Strategie A: Naives Gebot (Punkt-Forecast direkt)
$y = \hat{\omega}^{EN}$

### Strategie B: Newsvendor-korrigiertes Gebot
Da Windenergie asymmetrische Kosten hat (Unterproduktion teurer als Überproduktion):

$y^* = \hat{\omega}^{EN} + \Delta$

wobei $\Delta$ aus der Bias-Struktur des EN-Modells abgeleitet wird.


In [ ]:
# Einfaches Bidding: Punkt-Forecast + Newsvendor-Korrektur
# c_under: Kosten pro kWh Unterproduktion (reBAP - DA Preis)
# c_over:  Kosten pro kWh Überproduktion  (DA - reBAP Preis)

c_under = 15.0  # EUR/MWh – TODO: aus echten Preisdaten
c_over  = 10.0  # EUR/MWh – TODO: aus echten Preisdaten

tau = c_under / (c_under + c_over)  # optimaler Quantilwert
print(f"Newsvendor tau = {tau:.3f} → Gebot sollte {tau*100:.0f}%-Quantil entsprechen")

# Naive Strategie: Punkt-Forecast direkt
y_bid_naive = y_pred_aic.copy()

# Korrigierte Strategie: Quantilkorrektur über Residuen-Verteilung
residuals_train = y_train - en_aic.predict(X_tr_sc)
bias_correction = np.quantile(residuals_train, tau)
y_bid_corrected = y_pred_aic + bias_correction

print(f"Bias-Korrektur (Quantil der Residuen): {bias_correction:.2f} kW")

# Newsvendor Loss
def newsvendor_loss(y_true, y_bid, c_o, c_u):
    over  = np.maximum(y_bid - y_true, 0)
    under = np.maximum(y_true - y_bid, 0)
    return np.mean(c_o * over + c_u * under)

loss_naive     = newsvendor_loss(y_test, y_bid_naive,     c_over, c_under)
loss_corrected = newsvendor_loss(y_test, y_bid_corrected, c_over, c_under)

print(f"\nNewsvendor Loss – Naiv:      {loss_naive:.2f}")
print(f"Newsvendor Loss – Korrigiert: {loss_corrected:.2f}")
print(f"Verbesserung durch Korrektur: {(loss_naive - loss_corrected)/loss_naive*100:.1f}%")


## 8. Zusammenfassung

### Ergebnisse

| Methode | RMSE | MAE | Newsvendor Loss |
|---------|------|-----|----------------|
| EN (AIC) | s.o. | s.o. | s.o. |
| EN (CV)  | s.o. | s.o. | – |

### Key Insights

1. **Sparsität**: Elastic Net setzt irrelevante IMFs auf 0 → reduziert Overfitting
2. **Gruppenselektion**: Korrelierte IMFs werden gemeinsam behandelt (vs. LASSO: willkürliche Auswahl)
3. **AIC vs. CV**: AIC ist schneller, CV robuster – beide liefern ähnliche Ergebnisse
4. **Bidding**: EN als Punkt-Forecast + Residuen-Quantilkorrektur ≈ praxisnaher Ansatz

### Offene TODOs
- [ ] Echte Winddaten aus `data/processed/final_dataset.csv` laden
- [ ] Echte IMFs aus EMD-Zerlegung als Features nutzen (statt synthetische)
- [ ] Echte Preisdaten für Newsvendor Loss einsetzen
- [ ] Vergleich mit Quantile EN (direkte probabilistische Prognose)
